In [17]:
import pandas as pd

drop_events = pd.read_csv("../data/drop_events.csv")
drop_products = pd.read_csv("../data/drop_products.csv")
products = pd.read_csv("../data/products.csv")

drop_events.head()

,drop_id,retailer,source,zip_code,price_observed,observed_at,notes
0,1,Target,app,60491,49.99,2026-03-27T20:16:04.167692,ETB restock via Target app


In [18]:
store_activity = (
    drop_events[[
        "drop_id",
        "retailer",
        "zip_code",
        "price_observed",
        "observed_at"
    ]]
    .merge(drop_products, on="drop_id", how="left")
    .merge(
        products[[
            "product_id",
            "product_type"
        ]],
        on="product_id",
        how="left"
    )
)

store_activity.head()

,drop_id,retailer,zip_code,price_observed,observed_at,product_id,product_type
0,1,Target,60491,49.99,2026-03-27T20:16:04.167692,1,NaN


In [19]:
store_activity["store_key"] = (
    store_activity["retailer"].astype(str)
    + " | "
    + store_activity["zip_code"].astype(str)
)

In [20]:
store_summary = (
    store_activity
    .groupby(["retailer", "zip_code"])
    .agg(
        total_drops=("drop_id", "count"),
        unique_products=("product_id", "nunique"),
        avg_price=("price_observed", "mean"),
        last_seen=("observed_at", "max")
    )
    .reset_index()
    .sort_values("total_drops", ascending=False)
)

store_summary

,retailer,zip_code,total_drops,unique_products,avg_price,last_seen
0,Target,60491,1,1,49.99,2026-03-27T20:16:04.167692


In [21]:
store_summary["store_reliability_score"] = (
    store_summary["total_drops"] * 1.0
    + store_summary["unique_products"] * 0.5
)

store_summary.sort_values(
    "store_reliability_score",
    ascending=False
)

,retailer,zip_code,total_drops,unique_products,avg_price,last_seen,store_reliability_score
0,Target,60491,1,1,49.99,2026-03-27T20:16:04.167692,1.5


In [22]:
store_summary.to_csv("../data/store_patterns.csv", index=False)
print("✅ store_patterns.csv saved")
store_summary[["retailer", "zip_code", "total_drops"]]

✅ store_patterns.csv saved


,retailer,zip_code,total_drops
0,Target,60491,1
